Tải dữ liệu từ kaggle

In [ ]:
# Nạp thư viện kagglehub vào Python để sử dụng các hàm quản lý/tải dữ liệu từ Kaggle.
import kagglehub
# Tải phiên bản mới nhất của tập dữ liệu
path = kagglehub.dataset_download("nancyalaswad90/diamonds-prices")
# In ra màn hình dòng chữ thông báo kèm theo giá trị của biến path
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'diamonds-prices' dataset.
Path to dataset files: /kaggle/input/diamonds-prices


Đọc dữ liệu

In [ ]:
# Nạp thư viện os (Operating System) — dùng để tương tác với hệ điều hành (ví dụ: quản lý tệp tin, nối đường dẫn thư mục, kiểm tra tệp tồn tại).
import os
# Nạp thư viện numpy và đặt tên viết tắt là np — dùng để xử lý tính toán đại số tuyến tính, mảng nhiều chiều (array) và các phép toán số học tốc độ cao.
import numpy as np
# Nạp thư viện pandas và đặt tên viết tắt là pd — dùng để làm việc với dữ liệu dạng bảng (DataFrame), hỗ trợ đọc/ghi tệp CSV, Excel và tiền xử lý dữ liệu
import pandas as pd

# Kiểm tra nếu biến 'path' chưa tồn tại (ví dụ do chưa chạy ô trước hoặc restart kernel),
# thì tự động gán đường dẫn mặc định từ thư mục tải về của kagglehub.
try:
    path
except NameError:
    print("Cảnh báo: Không tìm thấy biến 'path'. Đang sử dụng đường dẫn mặc định...")
    path = "/root/.cache/kagglehub/datasets/nancyalaswad90/diamonds-prices/versions/4"

# Ghép đường dẫn giữa thư mục 'path' (từ kagglehub) và tên tệp CSV
file_path = os.path.join(path, "Diamonds Prices2022.csv")

# Đọc tệp CSV vào biến 'diamonds' (DataFrame) với dấu phân cách các cột là dấu phẩy (,)
diamonds = pd.read_csv(file_path, sep=',')

# In kích thước của tập dữ liệu (số dòng, số cột)
print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Hiển thị thông tin tổng quan
diamonds.info()

# In tiêu đề thông báo cho phần hiển thị 5 dòng đầu
print("\n--- 5 dòng đầu tiên ---")
# Hiển thị 5 dòng dữ liệu đầu tiên
print(diamonds.head())

# In tiêu đề thông báo cho phần kiểm tra các giá trị phân loại
print("\n--- Giá trị phân loại của các thuộc tính dạng chuỗi ---")
# Lấy và in danh sách các giá trị duy nhất (không trùng lặp) của cột 'cut' (vết cắt)
print(f"Cut: {diamonds['cut'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'color' (màu sắc)
print(f"Color: {diamonds['color'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'clarity' (độ tinh khiết)
print(f"Clarity: {diamonds['clarity'].unique()}")

# In tiêu đề thông báo cho phần thống kê mô tả
print("\n--- Thống kê mô tả ---")
# Tính toán các chỉ số thống kê cơ bản cho các cột số
print(diamonds.describe())

Kích thước tập dữ liệu ban đầu: (53943, 11)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53943 entries, 0 to 53942
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53943 non-null  int64  
 1   carat       53943 non-null  float64
 2   cut         53943 non-null  object 
 3   color       53943 non-null  object 
 4   clarity     53943 non-null  object 
 5   depth       53943 non-null  float64
 6   table       53943 non-null  float64
 7   price       53943 non-null  int64  
 8   x           53943 non-null  float64
 9   y           53943 non-null  float64
 10  z           53943 non-null  float64
dtypes: float64(6), int64(2), object(3)
memory usage: 4.5+ MB

--- 5 dòng đầu tiên ---
   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0    326  

Tiền xử lý dữ liệu

In [ ]:
# Nạp công cụ OrdinalEncoder từ thư viện scikit-learn để mã hóa các biến phân loại có thứ tự
from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
# Kiểm tra xem cột số thứ tự thừa 'Unnamed: 0' có tồn tại trong DataFrame hay không
if 'Unnamed: 0' in diamonds.columns:
    # Nếu có, xóa bỏ cột này khỏi tập dữ liệu
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ dữ liệu nhiễu (kích thước x, y, z <= 0)
# Lấy tổng số dòng dữ liệu ban đầu trước khi lọc
initial_rows = len(diamonds)

# Lọc giữ lại các dòng có cả 3 kích thước (x, y, z) đều lớn hơn 0 và tạo bản sao dữ liệu mới
diamonds = diamonds[(diamonds['x'] > 0) & (diamonds['y'] > 0) & (diamonds['z'] > 0)].copy()

# Tính số lượng dòng dữ liệu bị loại bỏ (bằng số dòng ban đầu trừ số dòng sau khi lọc)
invalid_dimensions = initial_rows - len(diamonds)

# In số lượng dòng nhiễu đã bị loại bỏ ra màn hình (thêm \n ở đầu để cách dòng trên)
print(f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu (x, y, z <= 0).")

# 3: Loại bỏ các dòng trùng lặp hoàn toàn
# Đếm tổng số dòng dữ liệu bị lặp lại hoàn toàn trong tập dữ liệu
num_duplicates = diamonds.duplicated().sum()

# Xóa bỏ các dòng trùng lặp, chỉ giữ lại 1 bản ghi đại diện
diamonds = diamonds.drop_duplicates()

# In số lượng dòng trùng lặp đã xóa ra màn hình (thêm \n ở đầu để cách với dòng trên)
print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa thứ tự (Ordinal Encoding) cho các thuộc tính phân loại
# Định nghĩa danh sách các mức độ chất lượng của vết cắt 'cut' theo thứ tự tăng dần (Fair là thấp nhất, Ideal là cao nhất)
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

# Định nghĩa danh sách màu sắc 'color' theo thứ tự tăng dần từ kém nhất đến tốt nhất (J -> D)
color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']

# Định nghĩa danh sách độ tinh khiết 'clarity' theo thứ tự tăng dần từ kém nhất đến tốt nhất (I1 -> IF)
clarity_categories = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# Khởi tạo đối tượng OrdinalEncoder với danh sách thứ tự đã thiết lập cho từng cột
encoder = OrdinalEncoder(categories=[cut_categories, color_categories, clarity_categories])

# chuyển đổi các cột dữ liệu dạng chữ (phân loại) thành dạng số dựa trên thứ tự đã định nghĩa
diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(diamonds[['cut', 'color', 'clarity']])

# 5: Kiểm tra dữ liệu sau tiền xử lý
# In kích thước mới của tập dữ liệu (số dòng, số cột) sau khi đã làm sạch và chuyển đổi
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

# In tiêu đề thông báo hiển thị 5 dòng đầu
print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")

# Hiển thị 5 dòng dữ liệu đầu tiên sau tiền xử lý
print(diamonds.head())


Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
0   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
1   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
2   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
3   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
4   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


### Đánh giá mô hình BayesianRidge bằng phương pháp 5-Fold Cross Validation

In [ ]:
# Nạp các thư viện cần thiết
import time
import numpy as np
import pandas as pd
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Chuẩn bị dữ liệu đầu vào (X) và nhãn dự đoán (y)
X = diamonds.drop(columns=['price'])
y = diamonds['price']

# 2. Định nghĩa lưới tham số cần tìm kiếm
max_iter_list = [100, 300, 500, 1000]
tol_list = [0.001, 0.0001, 0.00001]

# Khởi tạo các biến để theo dõi cấu hình tối ưu dựa trên R² trung bình của tập Test
best_r2_cv = -float('inf')
best_params_cv = {}
best_df_results_cv = None
best_total_time_cv = 0

# Cấu hình K-Fold cố định với k=5
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Duyệt qua các tổ hợp tham số của mô hình
for m_iter in max_iter_list:
    for tol_val in tol_list:
        results_list = []
        fold_idx = 1
        total_start = time.time()

        # Tiến hành chạy K-Fold cho từng tổ hợp tham số
        for train_idx, test_idx in kf.split(X):
            start_time = time.time()
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # Thiết lập mô hình với tham số hiện tại
            model = BayesianRidge(
                max_iter=m_iter,
                tol=tol_val,
                alpha_1=1e-6,
                alpha_2=1e-6,
                lambda_1=1e-6,
                lambda_2=1e-6,
                fit_intercept=True
            )
            model.fit(X_train, y_train)

            # Dự đoán
            y_train_pred = model.predict(X_train)
            y_test_pred = model.predict(X_test)

            # Tính toán nhanh các giá trị đánh giá và RMSE trực tiếp bằng toán học
            mse_train = mean_squared_error(y_train, y_train_pred)
            mse_test = mean_squared_error(y_test, y_test_pred)

            results_list.append({
                'Fold': f"Fold {fold_idx}",
                'Số mẫu Train': len(train_idx),
                'Số mẫu Test': len(test_idx),
                'MAE Train': mean_absolute_error(y_train, y_train_pred),
                'MSE Train': mse_train,
                'RMSE Train': np.sqrt(mse_train),
                'R² Train': r2_score(y_train, y_train_pred),
                'MAE Test': mean_absolute_error(y_test, y_test_pred),
                'MSE Test': mse_test,
                'RMSE Test': np.sqrt(mse_test),
                'R² Test': r2_score(y_test, y_test_pred),
                'Thời gian (giây)': time.time() - start_time
            })
            fold_idx += 1

        # Tổng hợp kết quả nhanh bằng Pandas
        df_temp = pd.DataFrame(results_list)
        mean_r2_test = df_temp['R² Test'].mean()

        # Nếu R² Test trung bình tốt hơn, lưu lại cấu hình
        if mean_r2_test > best_r2_cv:
            best_r2_cv = mean_r2_test
            best_params_cv = {'max_iter': m_iter, 'tol': tol_val}
            best_total_time_cv = time.time() - total_start

            # Thêm hàng trung bình vào bảng kết quả
            numeric_cols = df_temp.select_dtypes(include=[np.number]).columns
            mean_series = df_temp[numeric_cols].mean()
            mean_row = pd.Series({'Fold': 'Trung bình (Mean)'})
            mean_row = pd.concat([mean_row, mean_series])
            best_df_results_cv = pd.concat([df_temp, pd.DataFrame([mean_row])], ignore_index=True)

# 3. In kết quả tối ưu nhất của 5-Fold Cross Validation theo dạng bảng
print("=== KẾT QUẢ TỐT NHẤT 5-FOLD CROSS VALIDATION ===")
print(f"Bộ tham số tối ưu: n_iter (max_iter) = {best_params_cv['max_iter']}, tol = {best_params_cv['tol']}\n")
pd.set_option('display.float_format', lambda x: '%.4f' % x if not pd.isna(x) else '')
display(best_df_results_cv)
print(f"\nTổng thời gian huấn luyện cấu hình tốt nhất: {best_total_time_cv:.4f} giây")

=== KẾT QUẢ TỐT NHẤT 5-FOLD CROSS VALIDATION ===
Bộ tham số tối ưu: n_iter (max_iter) = 100, tol = 0.001



,Fold,Số mẫu Train,Số mẫu Test,MAE Train,MSE Train,RMSE Train,R² Train,MAE Test,MSE Test,RMSE Test,R² Test,Thời gian (giây)
0,Fold 1,43020.0000,10755.0000,799.7163,1474230.8842,1214.1791,0.9073,802.2033,1469448.4663,1212.2081,0.9070,0.0323
1,Fold 2,43020.0000,10755.0000,800.5179,1467202.5548,1211.2814,0.9081,797.6953,1498332.9728,1224.0641,0.9037,0.0329
2,Fold 3,43020.0000,10755.0000,797.9632,1465676.6263,1210.6513,0.9078,809.7096,1503717.8019,1226.2617,0.9052,0.0347
3,Fold 4,43020.0000,10755.0000,801.3353,1482011.5147,1217.3790,0.9062,800.2472,1439007.8169,1199.5865,0.9114,0.0310
4,Fold 5,43020.0000,10755.0000,799.0944,1473441.7215,1213.8541,0.9072,794.6993,1625164.7797,1274.8195,0.8982,0.0301
5,Trung bình (Mean),43020.0000,10755.0000,799.7254,1472512.6603,1213.4690,0.9073,800.9109,1507134.3675,1227.3880,0.9051,0.0322



Tổng thời gian huấn luyện cấu hình tốt nhất: 0.1674 giây
